# Guide 2 — Describing a cell: from materials to published record

This guide covers BattINFO's full cell-description workflow — from authoring individual electrode and electrolyte components, through building a complete cell specification, to saving and publishing the record.

**Estimated time:** 30 minutes
**Prerequisites:** [Guide 1 — Concepts](01-concepts.ipynb)

Everything writes under `.battinfo/notebooks/guide-02`.

In [1]:
import os, json, warnings
from pathlib import Path

_here = Path.cwd()
if (_here / "src" / "battinfo").exists():
    REPO = _here
elif (_here.parent.parent / "src" / "battinfo").exists():
    REPO = _here.parent.parent
    os.chdir(REPO)
else:
    REPO = _here

print(f"Repo root: {REPO}")

WORKSPACE = REPO / '.battinfo/notebooks/guide-02'
WORKSPACE.mkdir(parents=True, exist_ok=True)
print('Workspace:', WORKSPACE)

Repo root: c:\Users\simonc\Documents\Github-local\battery-genome\BattINFO
Workspace: c:\Users\simonc\Documents\Github-local\battery-genome\BattINFO\.battinfo\notebooks\guide-02


## Part 1 — Simple path (datasheet-level)

Use this when you have a datasheet and want to describe a commercial cell quickly.

In [2]:
from battinfo import CellType, publish

cell_type = CellType(
    manufacturer="Samsung SDI",
    model="INR21700-50E",
    format="cylindrical",
    chemistry="Li-ion",
    positive_electrode_basis="NMC",
    negative_electrode_basis="graphite",
    size_code="R21700",
    source={
        "type": "datasheet",
        "name": "INR21700-50E Product Specification Rev 1.0",
        "url": "https://example.com/samsung-inr21700-50e.pdf",
    },
    nominal_properties={
        "nominal_capacity":                      {"value": 5.0,   "unit": "Ah"},
        "nominal_voltage":                       {"value": 3.6,   "unit": "V"},
        "rated_energy":                          {"value": 18.0,  "unit": "Wh"},
        "mass":                                  {"value": 68.0,  "unit": "g"},
        "height":                                {"value": 70.15, "unit": "mm"},
        "diameter":                              {"value": 21.0,  "unit": "mm"},
        "maximum_continuous_charging_current":   {"value": 4.875, "unit": "A"},
        "maximum_continuous_discharging_current":{"value": 9.8,   "unit": "A"},
        "maximum_charging_temperature":          {"value": 45,    "unit": "°C"},
        "minimum_discharging_temperature":       {"value": -20,   "unit": "°C"},
        "cycle_life":                            {"value_text": ">500", "unit": "count"},
    },
)

result = publish(cell_type, destination="local", root=WORKSPACE)
print("IRI:", result.canonical_iri)

IRI: https://w3id.org/battinfo/cell/03yx-y1kk-q3mq-ck3w


In [3]:
# Inspect the canonical record
record = json.loads(Path(result.debug_paths["canonical_record_path"]).read_text(encoding="utf-8"))
print(json.dumps(record, indent=2))

{
  "schema_version": "1.0.0",
  "product": {
    "id": "https://w3id.org/battinfo/cell/03yx-y1kk-q3mq-ck3w",
    "short_id": "03yxy1",
    "identifier": "cell-type:03yx-y1kk-q3mq-ck3w",
    "name": "Samsung SDI INR21700-50E",
    "model": "INR21700-50E",
    "manufacturer": {
      "type": "Organization",
      "name": "Samsung SDI"
    },
    "cell_format": "cylindrical",
    "chemistry": "Li-ion",
    "positive_electrode_basis": "NMC",
    "negative_electrode_basis": "graphite",
    "size_code": "R21700"
  },
  "specs": {
    "nominal_capacity": {
      "value": 5.0,
      "unit": "Ah"
    },
    "nominal_voltage": {
      "value": 3.6,
      "unit": "V"
    },
    "rated_energy": {
      "value": 18.0,
      "unit": "Wh"
    },
    "mass": {
      "value": 68.0,
      "unit": "g"
    },
    "height": {
      "value": 70.15,
      "unit": "mm"
    },
    "diameter": {
      "value": 21.0,
      "unit": "mm"
    },
    "maximum_continuous_charging_current": {
      "value": 4.875,
  

In [4]:
# Produce the resolver JSON-LD
from battinfo.api import publish_record

output = publish_record(
    result.debug_paths["canonical_record_path"],
    target_root=WORKSPACE / "resolver",
)
jsonld = json.loads(Path(output["output_dir"], "index.jsonld").read_text(encoding="utf-8"))

print("@type:", jsonld["@type"])
print()
print("Properties:")
for prop in jsonld.get("hasProperty", []):
    t = prop["@type"][0] if isinstance(prop["@type"], list) else prop["@type"]
    v = prop.get("hasNumericalPart", {}).get("hasNumericalValue", "—")
    u = prop.get("hasMeasurementUnit", "").split("#")[-1]
    print(f"  {t}: {v} [{u}]")

@type: ['BatteryCellSpecification', 'schema:CreativeWork']

Properties:
  NominalCapacity: 5.0 [AmpereHour]
  NominalVoltage: 3.6 [Volt]
  battinfo:ratedEnergy: 18.0 [WattHour]
  Mass: 68.0 [Gram]
  Height: 70.15 [MilliMetre]
  Diameter: 21.0 [MilliMetre]
  MaximumContinuousChargingCurrent: 4.875 [Ampere]
  MaximumContinuousDischargingCurrent: 9.8 [Ampere]
  MaximumChargingTemperature: 45 [EMMO_36a9bf69_483b_42fd_8a0c_7ac9206320bc]
  MinimumDischargingTemperature: -20 [EMMO_36a9bf69_483b_42fd_8a0c_7ac9206320bc]
  CycleLife: — [EMMO_5ebd5e01_0ed3_49a2_a30d_cd05cbe72978]


## Part 2 — Descriptor path (material-level)

Use this for research cells or when electrode composition, electrolyte formulation, and construction details are known.
The workflow builds bottom-up: **materials → BOM → electrode → electrolyte → separator → cell description**.

### Step 1: Individual materials

In [5]:
from battinfo.authoring import material, properties

# Positive electrode
lfp   = material("LFP",          mass_fraction={"value": 90, "unit": "%"},
                  comment="LiFePO4, olivine, D50 ~ 3 µm")
pvdf  = material("PVDF",         mass_fraction={"value":  5, "unit": "%"})
c65   = material("Carbon black", mass_fraction={"value":  5, "unit": "%"},
                  comment="Imerys C65")

# Negative electrode
graphite = material("Graphite",  mass_fraction={"value": 95.5, "unit": "%"},
                     comment="Natural graphite, D50 ~ 20 µm")
cmc  = material("CMC", mass_fraction={"value": 1.5, "unit": "%"})
sbr  = material("SBR", mass_fraction={"value": 2.5, "unit": "%"})
c_ng = material("Carbon black", mass_fraction={"value": 0.5, "unit": "%"})

print("Materials defined.")

Materials defined.


> **Semantic note — `mass_fraction` as a compositional property**
>
> Strictly speaking, `mass_fraction` is not an intrinsic property of LFP or PVDF — it is a property of that material *as a constituent of this specific electrode coating*. The fully rigorous EMMO representation would attach the fraction to an intermediate constituent-role node rather than to the material node directly.
>
> BattINFO uses the simpler form — `mass_fraction` on the material node — as a deliberate authoring convenience. The ambiguity is resolved by context: every `material()` call here appears inside an electrode composition, so the fraction is always interpreted as the proportion of that material within the enclosing coating. A material node is never used in isolation where a context-free mass fraction would be meaningless.
>
> The same logic applies to `volume_fraction` on electrolyte solvent nodes in the electrolyte recipe below.

### Step 2: Bills of materials

In [6]:
from battinfo.authoring import bom

positive_bom = bom(active_material=lfp,      binder=pvdf,       additive=c65)
negative_bom = bom(active_material=graphite, binder=[cmc, sbr], additive=c_ng)

print("Positive active material:", positive_bom.active_material[0].name)
print("Negative binders:", [m.name for m in negative_bom.binder])

Positive active material: LFP
Negative binders: ['CMC', 'SBR']


### Step 3: Electrodes

In [7]:
from battinfo.authoring import electrode

positive_electrode = electrode(
    bom=positive_bom,
    loading={"value": 12.5, "unit": "mg/cm2"},
    calendered_density={"value": 2.4, "unit": "g/cm3"},
    current_collector="Aluminium foil",
    current_collector_thickness={"value": 16.0, "unit": "µm"},
    coating_comment="NMP-cast, roll-calendered",
)

negative_electrode = electrode(
    bom=negative_bom,
    loading={"value": 7.8, "unit": "mg/cm2"},
    calendered_density={"value": 1.55, "unit": "g/cm3"},
    current_collector="Copper foil",
    current_collector_thickness={"value": 10.0, "unit": "µm"},
)

print("Positive loading:", positive_electrode.coating.property.get("loading"))
print("Negative current collector:", negative_electrode.current_collector.name)

Positive loading: {'value': 12.5, 'unit': 'mg/cm2'}
Negative current collector: Copper foil


### Step 4: Electrolyte

In [8]:
from battinfo.authoring import electrolyte_recipe, material as mat

lp30 = electrolyte_recipe(
    family="organic",
    salt="LiPF6",
    salt_concentration={"value": 1.0, "unit": "mol/L"},
    solvents=[
        mat("EC",  volume_fraction={"value": 50, "unit": "%"}),
        mat("DMC", volume_fraction={"value": 50, "unit": "%"}),
    ],
    additives=[
        mat("VC",  mass_fraction={"value": 2.0, "unit": "%"}, comment="SEI-forming additive"),
    ],
    comment="LP30 + 2% VC",
)

print("Salt:", lp30.salt.name, "at", lp30.salt.property.get("concentration"))
print("Solvents:", [s.name for s in lp30.solvent_mixture.component])

Salt: LiPF6 at {'value': 1.0, 'unit': 'mol/L'}
Solvents: ['EC', 'DMC']


### Step 5: Separator and construction

In [9]:
from battinfo.authoring import separator_spec, construction

sep = separator_spec(
    material="Polypropylene",
    thickness={"value": 25.0, "unit": "µm"},
    properties=properties(porosity={"value": 41, "unit": "%"}),
    comment="Celgard 2500, single-layer PP",
)

build = construction(assembly_type="wound", layering="jelly-roll", layer_count=1)

print("Separator material:", sep.material)
print("Assembly:", build.assembly_type, "/", build.layering)

Separator material: Polypropylene
Assembly: wound / jelly-roll


### Step 6: Assemble the cell description

In [10]:
from battinfo.authoring import cell_description, source

# The id is the BattINFO IRI for this cell type.
# For new records: publish a CellType first to receive the IRI, then enrich it here.
# For this tutorial we use the known IRI for the A123 ANR26650M1-B example.
A123_IRI = "https://w3id.org/battinfo/cell/3m6k-9t2p-7x4h-9nq8"

spec = cell_description(
    id=A123_IRI,
    manufacturer="A123 Systems",
    model="ANR26650M1-B",
    format="cylindrical",
    chemistry="Li-ion",
    positive_electrode_basis="LFP",
    negative_electrode_basis="graphite",
    size_code="R26650",
    positive_electrode=positive_electrode,
    negative_electrode=negative_electrode,
    electrolyte=lp30,
    separator=sep,
    construction=build,
    source=source(
        type="datasheet",
        name="ANR26650M1-B Product Datasheet",
        url="https://example.com/a123-anr26650m1-b.pdf",
    ),
    comment="Detailed descriptor from manufacturer datasheet and published literature.",
)

print("Cell ID:", spec.id)
print("Format:", spec.format)

Cell ID: https://w3id.org/battinfo/cell/3m6k-9t2p-7x4h-9nq8
Format: cylindrical


### Step 7: Serialise and inspect the descriptor

In [11]:
# Write the descriptor dict to a JSON file for inspection and downstream use.
# The CellSpecification is the authoring object; publishing uses CellType.
desc_dir = WORKSPACE / "descriptors"
desc_dir.mkdir(parents=True, exist_ok=True)
desc_path = desc_dir / "a123-anr26650m1-b.descriptor.json"
desc_path.write_text(json.dumps(spec.model_dump(), indent=2, ensure_ascii=False), encoding="utf-8")
print("Descriptor written to:", desc_path.relative_to(WORKSPACE))

# Preview top-level keys
list(spec.model_dump().keys())

Descriptor written to: descriptors\a123-anr26650m1-b.descriptor.json


['schema_version',
 'kind',
 'id',
 'manufacturer',
 'model',
 'format',
 'chemistry',
 'positive_electrode_basis',
 'negative_electrode_basis',
 'size_code',
 'construction',
 'properties',
 'positive_electrode',
 'negative_electrode',
 'electrolyte',
 'separator',
 'coin_hardware',
 'specification_comment',
 'source',
 'comment']

### Export descriptor to domain-battery JSON-LD

The `CellSpecification` maps to EMMO-aligned JSON-LD via `to_jsonld(target="domain-battery")`. This is the rich semantic representation: it includes electrode composition nodes (constituents with mass fractions, loading, current collector), the electrolyte (salt, solvents, additives with concentrations and volume fractions), and the separator (material, thickness, porosity).

The descriptor pipeline expects the specification wrapped in a document envelope (`{"schema_version": ..., "specification": ...}`).

In [12]:
from battinfo.transform.json_to_jsonld import to_jsonld

descriptor_doc = json.loads(desc_path.read_text(encoding="utf-8"))

# The descriptor pipeline expects the specification wrapped in a document envelope.
# spec.model_dump() produces the specification fields directly;
# wrapping it here routes to the full descriptor pipeline.
desc_jsonld = to_jsonld(
    {"schema_version": descriptor_doc.get("schema_version"), "specification": descriptor_doc},
    target="domain-battery",
)

# Write to disk alongside the descriptor JSON
desc_jsonld_path = desc_path.with_suffix(".domain-battery.jsonld")
desc_jsonld_path.write_text(
    json.dumps(desc_jsonld, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"Domain-battery JSON-LD exported to: {desc_jsonld_path.relative_to(WORKSPACE)}")
print()
# Show key structural nodes
spec_node = desc_jsonld["@graph"][0]
pos_coating = spec_node["hasPositiveElectrode"]["hasCoating"]
elyte = spec_node["hasElectrolyte"]
print("@type:", spec_node["@type"])
print("isDescriptionFor:", spec_node["isDescriptionFor"]["@type"])
print("hasPositiveElectrode:", spec_node["hasPositiveElectrode"]["@type"])
print("  hasActiveMaterial:", pos_coating["hasActiveMaterial"]["@type"])
print("  hasBinder:", pos_coating["hasBinder"]["@type"])
print("  hasConductiveAdditive:", pos_coating["hasConductiveAdditive"]["@type"])
print("hasNegativeElectrode:", spec_node["hasNegativeElectrode"]["@type"])
print("hasElectrolyte:", elyte["@type"])
print("  hasSolute:", elyte["hasSolute"]["@type"])
solvents = elyte["hasSolvent"]
if not isinstance(solvents, list): solvents = [solvents]
print("  hasSolvent:", [s["@type"] for s in solvents])
print("hasSeparator:", spec_node["hasSeparator"]["@type"])


Domain-battery JSON-LD exported to: descriptors\a123-anr26650m1-b.descriptor.domain-battery.jsonld

@type: ['BatteryCellSpecification', 'schema:CreativeWork']
isDescriptionFor: ['BatteryCell', 'CylindricalBattery', 'LithiumIonBattery', 'LithiumIonIronPhosphateBattery', 'LithiumIonGraphiteBattery']
hasPositiveElectrode: LithiumIronPhosphateElectrode
  hasActiveMaterial: ['LithiumIronPhosphate', 'ActiveMaterial']
  hasBinder: ['PolyvinylideneFluoride', 'Binder']
  hasConductiveAdditive: ['CarbonBlack', 'ConductiveAdditive']
hasNegativeElectrode: GraphiteElectrode
hasElectrolyte: OrganicElectrolyte
  hasSolute: ['LithiumHexafluorophosphate', 'Solute']
  hasSolvent: [['EthyleneCarbonate', 'Solvent'], ['DimethylCarbonate', 'Solvent']]
hasSeparator: ['Polypropylene', 'Separator']


## Part 3 — Full linked graph

Combine the specification with a cell instance to produce the complete EMMO-aligned JSON-LD graph. This is the definitive semantic export: every node, predicate, and property value defined in Part 2 is present and verifiable here.

> **Note on the resolver artifact**: the lightweight JSON-LD served at the cell-type IRI is a discovery document, not the full graph. It carries `@type`, `schema:name`, and `hasProperty` for top-level specs, but omits electrode composition, electrolyte, and separator to keep file size small. Use the full graph below for semantic validation.

In [13]:
# Build the full linked graph: specification + one cell instance
# descriptor_doc was written in Step 7; reload it in case cells are run non-linearly.
descriptor_doc = json.loads(desc_path.read_text(encoding="utf-8"))

# In production, cell IRIs are minted by the registry at registration time.
# For verification we use a placeholder IRI that follows the BattINFO IRI scheme.
DEMO_CELL_IRI = "https://w3id.org/battinfo/cell/a123-anr26650m1-b-demo-001"

full_doc = {
    "schema_version": descriptor_doc.get("schema_version"),
    "specification": descriptor_doc,
    "instances": [
        {
            "id": DEMO_CELL_IRI,
            "name": "A123 ANR26650M1-B #demo-001",
            "serial_number": "demo-001",
            "comment": "Demonstration instance linked to the cell-type specification above.",
        }
    ],
}

full_jsonld = to_jsonld(full_doc, target="domain-battery")
full_jsonld_path = desc_dir / "a123-anr26650m1-b.linked.jsonld"
full_jsonld_path.write_text(
    json.dumps(full_jsonld, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"Full linked graph written to: {full_jsonld_path.relative_to(WORKSPACE)}")
print(f"Graph nodes: {len(full_jsonld['@graph'])}")
for i, node in enumerate(full_jsonld["@graph"]):
    node_type = node.get("@type", "?")
    node_id = node.get("@id", "(blank)")
    print(f"  [{i}] @type={node_type}  @id={node_id}")


Full linked graph written to: descriptors\a123-anr26650m1-b.linked.jsonld
Graph nodes: 2
  [0] @type=['BatteryCellSpecification', 'schema:CreativeWork']  @id=https://w3id.org/battinfo/cell/3m6k-9t2p-7x4h-9nq8
  [1] @type=['BatteryCell', 'schema:IndividualProduct']  @id=https://w3id.org/battinfo/cell/a123-anr26650m1-b-demo-001


In [14]:
# Print the full graph for verification
print(json.dumps(full_jsonld, indent=2, ensure_ascii=False))


{
  "@context": [
    "https://w3id.org/emmo/domain/battery/context",
    {
      "schema": "https://schema.org/"
    }
  ],
  "@graph": [
    {
      "@type": [
        "BatteryCellSpecification",
        "schema:CreativeWork"
      ],
      "isDescriptionFor": {
        "@type": [
          "BatteryCell",
          "CylindricalBattery",
          "LithiumIonBattery",
          "LithiumIonIronPhosphateBattery",
          "LithiumIonGraphiteBattery"
        ]
      },
      "@id": "https://w3id.org/battinfo/cell/3m6k-9t2p-7x4h-9nq8",
      "schema:name": "A123 Systems ANR26650M1-B",
      "schema:model": "ANR26650M1-B",
      "schema:manufacturer": {
        "@type": "schema:Organization",
        "schema:name": "A123 Systems"
      },
      "schema:size": "R26650",
      "hasPositiveElectrode": {
        "@type": "LithiumIronPhosphateElectrode",
        "hasCoating": {
          "@type": "ElectrodeCoating",
          "hasActiveMaterial": {
            "@type": [
              "Lithium

### Registry / Battery Genome

In [15]:
# Fill in real credentials to publish to a registry:
# result = publish(
#     cell_type,
#     destination="registry",          # or "battery-genome"
#     root=".battinfo/submissions",
#     registry_base_url="https://registry.example.org",
#     api_key="your-api-key",
#     workspace_id="my-lab",
#     publisher_id="sintef-industry",
#     source_version="2026-05-04",
# )
print("Registry publish shown commented out — fill in credentials to activate.")

Registry publish shown commented out — fill in credentials to activate.


## Next

- **[Guide 3 — Linked records](03-linked-records.ipynb)**: connect a cell instance, test, and dataset to the cell type and publish the chain